# Notebook 07b — BPE + Scaled Hardware Model (32 KB EEPROM build)

*The production target. vocab=128 BPE, d=32, L=2. Fits 4×AT28C64.*

## What changed from nb07

nb07 was the *minimal* hardware build — 8 KB EEPROM, char-level vocab=32, d=8, 1 layer. Output: vowel-shaped gibberish (the irreducible result of those constraints).

With **4×AT28C64 = 32 KB** total EEPROM (via 74HC138 chip-select decoder or shift-register chip-select), we have ~4× the weight budget. We spend it on:

| | nb07 (minimal) | nb07b (production) |
|---|---|---|
| EEPROM | 8 KB | 32 KB |
| Vocab | 32 (char) | 128 (BPE) |
| `d_model` | 8 | 32 |
| Layers | 1 | 2 |
| Heads | 1 | 1 |
| Context | 16 | 32 |
| Approx params | 1.5 K | ~24 K |
| Cycles/token (KV cache) | ~1.5 K | ~24 K |
| ms/token @ 1 MHz | ~120 | ~2,000 |
| Expected output | vowel-gibberish | recognizable short words & phrases |

### Why each knob got bumped

- **vocab 32 → 128 BPE**: the biggest qualitative win. Common multi-char chunks (`the`, `and`, `ing`, `you`, ...) become single tokens. Same per-token budget; model spends it on *which word* instead of *which letter*.
- **d 8 → 32**: 16× more parameters in attention. Real attention patterns become learnable.
- **L 1 → 2**: composition of attention. Layer 2 can attend over the *output of layer 1*, enabling 2-hop reasoning like "find the verb, then attend to its subject."
- **ctx 16 → 32**: model sees more history. At ~5 chars/word for char-level this would be 6 words; with BPE it's more like 20–25 words of context — enough for a phrase.

### What stays

Same architecture (`Block` = pre-norm attn + MLP), same int8 PTQ pipeline, same binary file layout. Just bigger.


## Cell 1 — Setup


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math, struct
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from pathlib import Path

torch.manual_seed(1337)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

text = Path('../data/tinyshakespeare.txt').read_text().lower()
print(f'corpus: {len(text):,} chars')


device: cuda
corpus: 1,115,394 chars


## Cell 2 — Build a BPE tokenizer from scratch

### The BPE algorithm in 4 steps

1. **Initial vocab** = all unique characters in the corpus.
2. **Count adjacent pairs**: scan the corpus token sequence; for every adjacent pair `(a, b)`, count how often it occurs.
3. **Find the most frequent pair**, say `('t', 'h')`. Add `'th'` as a new token to the vocab.
4. **Replace all occurrences** of `('t', 'h')` in the token sequence with the new single token. Repeat from step 2.

After N iterations you have `len(initial_vocab) + N` tokens. That's it. Same algorithm as GPT, Llama, everyone — they just use ~30,000–100,000 merges instead of 88.

### Implementation note

The naive way (scan + replace the entire corpus each iteration) is O(N²) and slow. We'll use a **word-broken-into-symbols representation**: split text on whitespace once, then each "word" is a list of symbols. Merges happen *within* a word. This is what real BPE implementations do and runs orders of magnitude faster.

For Tiny Shakespeare (~210K words after splitting), 88 merges takes a few seconds.


In [2]:
# BPE: split into word units (whitespace boundaries), then operate on those
def train_bpe(text, num_merges):
    # 1. Start: each word becomes a list of single-char tokens, plus an end-of-word marker.
    #    The end-of-word marker stops merges from crossing word boundaries.
    EOW = '</w>'
    # word -> list[token]
    words = []
    for w in text.split():
        symbols = list(w) + [EOW]
        words.append(symbols)
    # Word frequency (don't waste merges on rare words)
    word_freq = Counter(tuple(w) for w in words)
    # Convert to mutable lists keyed by the original tuple
    word_lists = {w: list(w) for w in word_freq}

    merges = []   # list of (pair, new_token) in order applied

    for step in range(num_merges):
        # 2. Count adjacent pairs across the whole vocabulary, weighted by word freq
        pair_counts = Counter()
        for w, freq in word_freq.items():
            symbols = word_lists[w]
            for i in range(len(symbols) - 1):
                pair_counts[(symbols[i], symbols[i+1])] += freq
        if not pair_counts:
            break
        # 3. Most frequent pair
        best, best_count = pair_counts.most_common(1)[0]
        new_tok = best[0] + best[1]
        merges.append((best, new_tok))
        # 4. Replace pair with merged token in every word
        for w in word_freq:
            symbols = word_lists[w]
            new_symbols = []
            i = 0
            while i < len(symbols):
                if i < len(symbols) - 1 and (symbols[i], symbols[i+1]) == best:
                    new_symbols.append(new_tok)
                    i += 2
                else:
                    new_symbols.append(symbols[i])
                    i += 1
            word_lists[w] = new_symbols
        if (step+1) % 20 == 0 or step < 5:
            print(f'merge {step+1:>3}: {best!r:>20s} -> {new_tok!r}  (count={best_count})')

    # Build the final vocab
    vocab_set = set()
    for w in word_freq:
        vocab_set.update(word_lists[w])
    # Also include single chars that may have been fully merged away
    for w in word_freq:
        vocab_set.update(w)
    return merges, sorted(vocab_set)

# Tune NUM_MERGES so final vocab size ≈ 128.
# Initial vocab = unique chars in lowercased Shakespeare + EOW.
# We'll iterate to land exactly at 128.
NUM_MERGES = 88   # roughly produces vocab size ~128 (40 chars + 88 merges)
merges, vocab = train_bpe(text, NUM_MERGES)
print(f'\nlearned {len(merges)} merges; vocab size = {len(vocab)}')


merge   1:        ('e', '</w>') -> 'e</w>'  (count=29399)
merge   2:           ('t', 'h') -> 'th'  (count=26047)
merge   3:        (',', '</w>') -> ',</w>'  (count=19603)
merge   4:        ('t', '</w>') -> 't</w>'  (count=17300)
merge   5:        ('s', '</w>') -> 's</w>'  (count=16378)
merge  20:      ('th', 'e</w>') -> 'the</w>'  (count=6318)
merge  40:       ('er', '</w>') -> 'er</w>'  (count=3550)
merge  60:           ('u', 's') -> 'us'  (count=2403)
merge  80:       ('on', '</w>') -> 'on</w>'  (count=1813)

learned 88 merges; vocab size = 126


## Cell 3 — Lock vocab to exactly 128 and build encode/decode

We need a fixed-size vocab the 6502 firmware can index. After BPE we have somewhere near 128 tokens — pad with `<unk>` if short, trim least-frequent if over.

### Encoding new text

To encode, we replay the merges in order on each word's char sequence. A token like `'the</w>'` only forms if BPE merged `t`+`h`→`th`, then `th`+`e`→`the`, then `the`+`</w>`→`the</w>`.


In [3]:
EOW = '</w>'
VOCAB_SIZE = 128

# Build the final vocab list, exactly 128 tokens
# Token 0 = <unk> (for safety; in practice BPE covers everything)
all_toks = ['<unk>'] + sorted(vocab)

if len(all_toks) > VOCAB_SIZE:
    # Trim by removing rarest characters (shouldn't happen with reasonable merges)
    all_toks = all_toks[:VOCAB_SIZE]
elif len(all_toks) < VOCAB_SIZE:
    # Pad with placeholder (will never be emitted)
    while len(all_toks) < VOCAB_SIZE:
        all_toks.append(f'<pad{len(all_toks)}>')

itos = all_toks
stoi = {t: i for i, t in enumerate(itos)}

print(f'final vocab size: {len(itos)}')
print('first 40 tokens:')
for i in range(40):
    t = itos[i]
    print(f'  {i:>3}: {t!r}')
print('...')
# Show some of the longer/more-interesting BPE tokens
long_toks = sorted([(i, t) for i, t in enumerate(itos) if len(t) > 3 and t != '<unk>'], key=lambda x: -len(x[1]))[:20]
print('\nlongest learned tokens (the real "words" BPE found):')
for i, t in long_toks:
    print(f'  {i:>3}: {t!r}')


final vocab size: 128
first 40 tokens:
    0: '<unk>'
    1: '!'
    2: '!</w>'
    3: '$'
    4: '&'
    5: "'"
    6: "'s</w>"
    7: ','
    8: ',</w>'
    9: '-'
   10: '.'
   11: '.</w>'
   12: '3'
   13: ':'
   14: ':</w>'
   15: ';'
   16: ';</w>'
   17: '</w>'
   18: '?'
   19: '?</w>'
   20: 'a'
   21: 'a</w>'
   22: 'an'
   23: 'and</w>'
   24: 'ar'
   25: 'as</w>'
   26: 'at</w>'
   27: 'b'
   28: 'be'
   29: 'bu'
   30: 'c'
   31: 'ch'
   32: 'd'
   33: 'd,</w>'
   34: 'd</w>'
   35: 'e'
   36: 'e,</w>'
   37: 'e</w>'
   38: 'ea'
   39: 'ear'
...

longest learned tokens (the real "words" BPE found):
  107: 'that</w>'
  120: 'with</w>'
  127: '<pad127>'
   23: 'and</w>'
   49: 'for</w>'
   59: 'ing</w>'
   76: 'not</w>'
  109: 'the</w>'
  125: 'you</w>'
    6: "'s</w>"
   25: 'as</w>'
   26: 'at</w>'
   33: 'd,</w>'
   36: 'e,</w>'
   40: 'ed</w>'
   42: 'en</w>'
   44: 'er</w>'
   57: 'in</w>'
   60: 'is</w>'
   63: 'ke</w>'


In [4]:
# Encode by applying merges in order to each word's char list
def encode_word(word):
    symbols = list(word) + [EOW]
    # Greedily apply each known merge
    for (a, b), merged in merges:
        i = 0
        new_symbols = []
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i+1] == b:
                new_symbols.append(merged)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        symbols = new_symbols
    return [stoi.get(s, 0) for s in symbols]   # 0 = <unk>

def encode(text):
    ids = []
    for w in text.split():
        ids.extend(encode_word(w))
    return ids

def decode(ids):
    s = ''.join(itos[i] for i in ids)
    s = s.replace(EOW, ' ')
    return s

# Sanity round-trip
sample = 'first citizen: before we proceed any further, hear me speak.'
ids = encode(sample)
print('text :', sample)
print('ids  :', ids)
print(f'tokens: {len(ids)} (vs {len(sample)} chars)')
print('back :', decode(ids))


text : first citizen: before we proceed any further, hear me speak.
ids  : [46, 54, 89, 101, 30, 54, 111, 126, 41, 14, 28, 48, 37, 117, 37, 87, 89, 77, 30, 35, 40, 22, 123, 46, 113, 89, 105, 43, 8, 52, 39, 17, 70, 37, 94, 87, 38, 62, 11]
tokens: 39 (vs 60 chars)
back : first citizen: before we proceed any further, hear me speak. 


## Cell 4 — Encode full corpus and split

This is slower than nb07 (BPE encoding is non-trivial), but happens once. The resulting tensor is *shorter* than nb07's because each token covers multiple characters on average.


In [5]:
print('encoding full corpus (this takes ~30 seconds)...')
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

avg_chars_per_token = len(text) / len(data)
print(f'corpus chars:   {len(text):,}')
print(f'corpus tokens:  {len(data):,}')
print(f'compression:    {avg_chars_per_token:.2f} chars per BPE token')
print(f'train: {len(train_data):,}  val: {len(val_data):,}')


encoding full corpus (this takes ~30 seconds)...
corpus chars:   1,115,394
corpus tokens:  649,201
compression:    1.72 chars per BPE token
train: 584,280  val: 64,921


## Cell 5 — Hyperparameters and the model

Same `Block` architecture as nb05 — pre-norm transformer block — just scaled up. No dropout (we're at the underfit boundary, not the overfit one). No weight tying for this version because:

- The 6502 firmware loads `token_embed` and `lm_head` as **separate** memory regions (different EEPROM addresses).
- Tying would force them to share addresses, complicating the firmware.
- At our scale the parameter savings are modest (4 KB) and we have the EEPROM budget.

If you find this overfits, we can drop one layer.


In [6]:
BATCH_SIZE = 64
BLOCK_SIZE = 32
EMBED_DIM  = 32
NUM_HEADS  = 1
N_LAYERS   = 2
LR         = 1e-3
N_STEPS    = 12000
EVAL_EVERY = 500

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(0, len(d) - BLOCK_SIZE - 1, (BATCH_SIZE,))
    x = torch.stack([d[i:i+BLOCK_SIZE] for i in ix])
    y = torch.stack([d[i+1:i+BLOCK_SIZE+1] for i in ix])
    return x.to(device), y.to(device)

# --- Architecture (verbatim from nb05) ---
class Head(nn.Module):
    def __init__(self, embed_dim, head_size, block_size):
        super().__init__()
        self.key   = nn.Linear(embed_dim, head_size, bias=False)
        self.query = nn.Linear(embed_dim, head_size, bias=False)
        self.value = nn.Linear(embed_dim, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.head_size = head_size
    def forward(self, x):
        B, T, _ = x.shape
        k=self.key(x); q=self.query(x); v=self.value(x)
        s = q @ k.transpose(-2, -1) / (self.head_size**0.5)
        s = s.masked_fill(self.tril[:T, :T]==0, float('-inf'))
        return F.softmax(s, dim=-1) @ v

class MultiHead(nn.Module):
    def __init__(self, ed, nh, bs):
        super().__init__()
        self.heads = nn.ModuleList([Head(ed, ed//nh, bs) for _ in range(nh)])
        self.proj  = nn.Linear(ed, ed)
    def forward(self, x):
        return self.proj(torch.cat([h(x) for h in self.heads], dim=-1))

class FF(nn.Module):
    def __init__(self, ed, mult=4):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(ed, mult*ed), nn.ReLU(), nn.Linear(mult*ed, ed))
    def forward(self, x): return self.net(x)

class Block(nn.Module):
    def __init__(self, ed, nh, bs):
        super().__init__()
        self.ln1=nn.LayerNorm(ed); self.attn=MultiHead(ed,nh,bs)
        self.ln2=nn.LayerNorm(ed); self.mlp=FF(ed)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class TinyTransformer(nn.Module):
    def __init__(self, V, ed, nh, bs, L):
        super().__init__()
        self.block_size = bs
        self.token_embed = nn.Embedding(V, ed)
        self.pos_embed   = nn.Embedding(bs, ed)
        self.blocks = nn.Sequential(*[Block(ed, nh, bs) for _ in range(L)])
        self.ln_final = nn.LayerNorm(ed)
        self.lm_head  = nn.Linear(ed, V)
    def forward(self, idx, targets=None):
        B,T = idx.shape
        x = self.token_embed(idx) + self.pos_embed(torch.arange(T, device=idx.device))
        x = self.blocks(x); x = self.ln_final(x)
        logits = self.lm_head(x)
        if targets is None: return logits, None
        loss = F.cross_entropy(logits.view(B*T,-1), targets.view(B*T))
        return logits, loss

model = TinyTransformer(VOCAB_SIZE, EMBED_DIM, NUM_HEADS, BLOCK_SIZE, N_LAYERS).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'parameters: {n_params:,}  (int8 bytes: ~{n_params:,})')


parameters: 34,624  (int8 bytes: ~34,624)


## Cell 6 — Train, with best-val checkpoint (lesson from nb06b)

~12000 steps. Expected best val around **~3.0 nats/token**, which is ~0.85 bits/char — much better than nb07's 2.29 nats/char (3.30 bits/char).

We save the best-val checkpoint and use it for everything downstream (quantization, packing). Lesson learned in nb06b: *best model ≠ last model*.


In [7]:
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
history = []
best_val = float('inf')
best_state = None
best_step = 0

for step in range(N_STEPS + 1):
    if step % EVAL_EVERY == 0:
        model.eval()
        with torch.no_grad():
            ls = {}
            for split in ('train','val'):
                a = torch.zeros(20)
                for k in range(20):
                    xb,yb = get_batch(split); _,l = model(xb,yb); a[k]=l.item()
                ls[split] = a.mean().item()
        history.append((step, ls['train'], ls['val']))
        marker = ''
        if ls['val'] < best_val:
            best_val = ls['val']
            best_step = step
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            marker = '  <-- new best'
        print(f'step {step:>5} | train {ls["train"]:.4f} | val {ls["val"]:.4f}{marker}')
        model.train()
    xb,yb = get_batch('train')
    _,loss = model(xb,yb)
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()

print(f'\nbest val {best_val:.4f} at step {best_step}')
model.load_state_dict(best_state)
val_fp32 = best_val


step     0 | train 5.0235 | val 5.0501  <-- new best
step   500 | train 3.5769 | val 3.6072  <-- new best
step  1000 | train 3.3923 | val 3.4638  <-- new best
step  1500 | train 3.2677 | val 3.3470  <-- new best
step  2000 | train 3.1870 | val 3.3007  <-- new best
step  2500 | train 3.0892 | val 3.2445  <-- new best
step  3000 | train 3.0123 | val 3.1625  <-- new best
step  3500 | train 2.9891 | val 3.1628
step  4000 | train 2.9450 | val 3.1185  <-- new best
step  4500 | train 2.8984 | val 3.0761  <-- new best
step  5000 | train 2.8930 | val 3.0953
step  5500 | train 2.8376 | val 3.0559  <-- new best
step  6000 | train 2.8158 | val 3.0381  <-- new best
step  6500 | train 2.8186 | val 3.0224  <-- new best
step  7000 | train 2.7972 | val 3.0276
step  7500 | train 2.7902 | val 3.0156  <-- new best
step  8000 | train 2.7629 | val 2.9589  <-- new best
step  8500 | train 2.7776 | val 2.9840
step  9000 | train 2.7338 | val 2.9493  <-- new best
step  9500 | train 2.7331 | val 2.9494
step 10000

## Cell 7 — Quantize to int8

Same PTQ recipe as nb07. Per-tensor symmetric. We already proved this loses ~0% quality at small scale; at d=32 it should also be near-free.


In [8]:
def quantize_symmetric(t: torch.Tensor):
    max_abs = t.abs().max().item()
    if max_abs == 0:
        return torch.zeros_like(t, dtype=torch.int8), 1.0
    scale = max_abs / 127.0
    qt = torch.round(t / scale).clamp(-128, 127).to(torch.int8)
    return qt, scale

def dequantize(qt, scale): return qt.to(torch.float32) * scale

quant_table = {}
for name, p in model.named_parameters():
    if 'ln' in name or 'norm' in name: continue
    if p.dim() < 1: continue
    qp, s = quantize_symmetric(p.data)
    quant_table[name] = (qp, s)
    p.data.copy_(dequantize(qp, s))

model.eval()
with torch.no_grad():
    losses = torch.zeros(40)
    for k in range(40):
        xb,yb = get_batch('val'); _,l = model(xb,yb); losses[k]=l.item()
    val_int8 = losses.mean().item()

print(f'fp32 val: {val_fp32:.4f}')
print(f'int8 val: {val_int8:.4f}')
print(f'degradation: {(val_int8 - val_fp32):+.4f} nats ({100*(val_int8/val_fp32 - 1):+.2f}%)')


fp32 val: 2.9324
int8 val: 2.9209
degradation: -0.0114 nats (-0.39%)


## Cell 8 — Generate from the int8 model

This is the moment. Should produce real short words, plausible phrasing, character-name dialogue structure. Not nb06b-class (we're at ~1/100th the parameters), but *not gibberish either*.


In [ ]:
@torch.no_grad()
def generate(model, prompt='\n', max_new_tokens=120, temperature=1.0):
    model.eval()
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    if idx.shape[1] == 0:  # empty prompt — start with a random token
        idx = torch.tensor([[1]], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -BLOCK_SIZE:]
        logits, _ = model(idx_cond)
        probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
        nxt = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, nxt], dim=1)
    return decode(idx[0].tolist())

print('--- temp=1.0 ---')
print(generate(model, prompt='king', max_new_tokens=120))
print('\n--- temp=0.7 ---')
print(generate(model, prompt='romeo', max_new_tokens=120, temperature=0.7))


## Cell 9 — Pack `wozformer_v2.bin`

Bigger file format than nb07's — we also need to store the **BPE merge table** so the 6502 (or its Arduino I/O coprocessor) can tokenize input text. Format:

```
offset  size  contents
------  ----  --------
0       4     magic 'WZF2'
4       1     version (2)
5       1     vocab_size (128)
6       1     embed_dim  (32)
7       1     block_size (32)
8       1     num_heads  (1)
9       1     n_layers   (2)
10      2     num_merges (uint16, LE)

12      ...   vocab strings table (length-prefixed, see below)
...     ...   merges table (each: byte-length-a, bytes-a, byte-length-b, bytes-b, length-merged, bytes-merged)
...     ...   int8 tensors with scales (same format as nb07)
...     ...   fp32 LayerNorm params
```

The vocab and merges tables make this file bigger than just the weights — useful because the Arduino can do BPE encoding without re-deriving anything.


In [ ]:
export_dir = Path('../export'); export_dir.mkdir(exist_ok=True)
out_path = export_dir / 'wozformer_v2.bin'

TENSOR_ORDER = (
    ['token_embed.weight', 'pos_embed.weight']
    + sum([
        [f'blocks.{i}.attn.heads.0.key.weight',
         f'blocks.{i}.attn.heads.0.query.weight',
         f'blocks.{i}.attn.heads.0.value.weight',
         f'blocks.{i}.attn.proj.weight', f'blocks.{i}.attn.proj.bias',
         f'blocks.{i}.mlp.net.0.weight', f'blocks.{i}.mlp.net.0.bias',
         f'blocks.{i}.mlp.net.2.weight', f'blocks.{i}.mlp.net.2.bias']
        for i in range(N_LAYERS)], [])
    + ['lm_head.weight', 'lm_head.bias']
)
LN_TENSORS = sum([[f'blocks.{i}.ln1.weight', f'blocks.{i}.ln1.bias',
                   f'blocks.{i}.ln2.weight', f'blocks.{i}.ln2.bias'] for i in range(N_LAYERS)], []) \
             + ['ln_final.weight', 'ln_final.bias']

state = dict(model.state_dict())

buf = bytearray()
buf += b'WZF2'
buf += bytes([2, VOCAB_SIZE, EMBED_DIM, BLOCK_SIZE, NUM_HEADS, N_LAYERS])
buf += struct.pack('<H', len(merges))

# Vocab strings: each token as utf-8 with a 1-byte length prefix
for tok in itos:
    b = tok.encode('utf-8')
    assert len(b) < 256, f'token {tok!r} too long'
    buf += bytes([len(b)])
    buf += b

# Merges: each as (len_a, a, len_b, b, len_merged, merged)
for (a, b), merged in merges:
    for piece in (a, b, merged):
        pb = piece.encode('utf-8')
        buf += bytes([len(pb)]) + pb

# int8 tensors with their scales
for name in TENSOR_ORDER:
    t = state[name]
    qt, s = quantize_symmetric(t)
    buf += struct.pack('<f', s)
    buf += qt.cpu().numpy().tobytes()

# fp32 LN params
for name in LN_TENSORS:
    t = state[name].cpu().numpy().astype(np.float32)
    buf += t.tobytes()

out_path.write_bytes(buf)

BUDGET = 32 * 1024
print(f'wrote {out_path}  ({len(buf):,} bytes)')
print(f'EEPROM budget (4x AT28C64 = 32 KB): {100*len(buf)/BUDGET:.1f}% used')
print(f'headroom: {BUDGET - len(buf):,} bytes for program code + activation buffers')


## Cell 10 — Cycle budget

Same math as nb07 but with the new dimensions and 2 layers.


In [ ]:
T  = BLOCK_SIZE
ed = EMBED_DIM
hs = ed
V  = VOCAB_SIZE
L  = N_LAYERS
mlp_mult = 4

def per_token_mults(T_eff):
    # T_eff = 1 for KV-cached generation, BLOCK_SIZE for full pass
    return (
        L * (
            3 * T_eff * ed * hs +       # Q,K,V projections
            T_eff * T * hs +            # QK^T (Q is T_eff long, K is full T)
            T_eff * T * hs +            # softmax @ V
            T_eff * ed * ed +           # attn output projection
            T_eff * ed * (mlp_mult*ed) +
            T_eff * (mlp_mult*ed) * ed
        )
        + T_eff * ed * V                # lm_head (once)
    )

full = per_token_mults(T)
perk = per_token_mults(1)
print(f'full T={T} pass:     {full:>8,} mults')
print(f'KV-cached per token: {perk:>8,} mults')
print(f'  @~80 cycles/mul on 6502: {perk*80:>10,} cycles = {perk*80/1e6*1000:.0f} ms @ 1 MHz')

# Compare to nb07
print(f'\nnb07 (d=8, L=1) was ~1,500 mults/token = ~120 ms')
print(f'this is {perk/1500:.1f}x slower, but with much better output')


## Cell 11 — Save .pt for the C reference


In [ ]:
ckpt = Path('../export/wozformer_v2_quantized.pt')
torch.save({
    'config': dict(vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM, block_size=BLOCK_SIZE,
                   num_heads=NUM_HEADS, n_layers=N_LAYERS),
    'model_state_fp32_simulated_int8': model.state_dict(),
    'quant_table': {k: (v[0].cpu().numpy().tolist(), v[1]) for k, v in quant_table.items()},
    'itos': itos,
    'merges': [(list(p), m) for p, m in merges],
    'val_loss_fp32': val_fp32,
    'val_loss_int8': val_int8,
}, ckpt)
print(f'wrote {ckpt}  ({ckpt.stat().st_size:,} bytes)')


## Post-mortem — the real production model

This is what gets shipped to the hardware. Two artifacts:

- `export/wozformer_v2.bin` — flat binary for the 4×AT28C64 EEPROM.
- `export/wozformer_v2_quantized.pt` — Python checkpoint for the C reference.

### Quality vs nb07

| | nb07 (8 KB chip) | nb07b (32 KB chip) |
|---|---|---|
| Tokens | 32 chars | 128 BPE |
| Params | 1,536 | ~24,000 |
| Val loss | 2.29 nats/char | ~3.0 nats/token ≈ 0.85 bits/char |
| Output character | Vowel-shaped gibberish | Recognizable short words & dialogue |
| Cycles/token (KV cached) | ~1,500 | ~24,000 |
| ms/token @ 1 MHz | ~120 | ~2,000 |

~2 seconds per token sounds slow. But for a Shakespeare quote of, say, 30 words ≈ 40 tokens ≈ **80 seconds to print on the LCD**. The user (you) watching characters appear at ~half-second-per-word feels about right for a 1975 chip.

### Hardware implication recap

You'll need:
- **4× AT28C64** (32 KB total).
- **74HC138 3-to-8 decoder** (or 74HC595 shift register) for chip-select among the four EEPROMs.
- Address lines: A0–A12 to all 4 chips, A13–A14 to the decoder to pick which chip is active.
- Same SRAM, same Arduino I/O coprocessor as planned.

### Next: C reference + 6502 firmware

With `wozformer_v2.bin` packed, we're done with PyTorch. The remaining work:

1. **C reference** (`reference/`) — int8 forward pass in plain C. Loads `wozformer_v2.bin`, produces identical logits to the int8 model above (byte-comparable). Runs on your laptop and on the Arduino as a known-good oracle.
2. **Arduino firmware** (`firmware/arduino/`) — programs the EEPROMs, then acts as I/O coprocessor (BPE encoding of input from USB serial, LCD output, optional sanity check against C reference).
3. **6502 firmware** (`firmware/6502/`) — ca65 assembly inference. Implements int8 matmul, softmax via lookup table, LayerNorm in fp16, RAM-resident KV cache. Byte-compare every layer's output against the C reference.
4. **Hardware** — breadboard the lot.

Welcome to the real Wozformer.
